In [ ]:
import os

import torch
torch.set_float32_matmul_precision('high')
torch.set_default_device('cuda')

from torch.utils.data import DataLoader
from torch.nn import MSELoss

from IPython.display import display, Audio

from tqdm import tqdm

from ddsp.params_vae import ParamsVAE
from ddsp import AudioDataset

dataset_path = '/mnt/mariadata/datasets/emf-dataset/processed'

fs = 44100
batch_size = 16
n_signal = fs * 2.0
n_sines = 8
batch_size = 32

dataset = AudioDataset(dataset_path=dataset_path, n_signal=n_signal, sampling_rate=fs, n_sines=n_sines)
loader = DataLoader(dataset=dataset, batch_size=batch_size)

model = ParamsVAE(n_params=n_sines*2, latent_size=4)

In [ ]:
epochs = 10000

# setup optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5000, threshold=1e-5, verbose=True)
loss_fn = MSELoss()

# create the progress bar
progress_bar = tqdm(total=epochs * len(loader), desc='Training Progress')

model.train()
for epoch in range(epochs):
  for batch in loader:
    model.zero_grad()
    audio, params = batch
    params = params.permute(0, 2, 1)

    # Forward pass
    output = model(params)

    # Compute loss (example: MSE)
    loss = loss_fn(output, params)

    # Backward pass
    loss.backward()
    optimizer.step()
    scheduler.step(loss)

    progress_bar.set_postfix({'loss': loss.item()})
    progress_bar.update(1)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import random

sns.set_style(style='darkgrid')

idx = random.randint(0, len(dataset) - 1)
audio, params = dataset[idx]
params = params.permute(1, 0).unsqueeze(0)  # Reshape params to match model input

model.eval()

with torch.no_grad():
  params_pred = model(params)

for i in range(n_sines*2):
  plt.figure(figsize=(10, 5))
  sns.lineplot(params[0, :, i].cpu().numpy(), label=f'Param {i}', alpha=0.5)
  sns.lineplot(params_pred[0, :, i].cpu().numpy(), label=f'Predicted Param {i}', linestyle='--', alpha=0.5)
  plt.xlim(100, 300)
  plt.show()
